In [2]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [4]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [5]:
from sklearn.model_selection import train_test_split

X = data['text']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 800
Test set size: 200


## Data Preprocessing

In [6]:
import string
import nltk

nltk.download('stopwords')

from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Jawad\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [7]:
import re

def clean_html(html_str):
  # Remove inline CSS / JS
  cleaned = re.sub(r"(?is)<(script|style).*?>.*?(</\1>)", " ", html_str)
  # Remove the HTML Comments
  cleaned = re.sub(r"(?s)<!--(.*?)-->[\n]?", " ", cleaned)
  # Remove remaining HTML Tags
  cleaned = re.sub(r"(?s)<(.*?)>", " ", cleaned)
  #Standardize the whitespace
  cleaned = re.sub(r"\s", " ", cleaned.strip())
  return cleaned

# Apply the HTML cleaning function to both partitions
X_train = X_train.apply(clean_html)
X_test = X_test.apply(clean_html)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [8]:
def clean_text(text):
    # Remove all the special characters
    text = re.sub(r'\W', ' ', str(text))
    # Remove numbers
    text = re.sub(r'\d+', ' ', text)
    # Remove all single characters
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)
    # Remove single characters from the start
    text = re.sub(r'^[a-zA-Z]\s+', ' ', text)
    # Substitute multiple spaces with single space
    text = re.sub(r'\s+', ' ', text, flags=re.I)
    # Remove prefixed 'b'
    text = re.sub(r'^b\s+', '', text)
    # Convert to Lowercase
    text = text.lower()
    return text

# Apply the text cleaning function to both partitions
X_train = X_train.apply(clean_text)
X_test = X_test.apply(clean_text)

## Now let's work on removing stopwords
Remove the stopwords.

In [9]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

X_train = X_train.apply(remove_stopwords)
X_test = X_test.apply(remove_stopwords)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [10]:
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

X_train = X_train.apply(lemmatize_text)
X_test = X_test.apply(lemmatize_text)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Jawad\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [11]:
from collections import Counter

# Recombine text and labels temporarily to filter by ham/spam
eda_df = pd.DataFrame({'text': X_train, 'label': y_train})

# Extract and count words for HAM (label == 0)
ham_words = ' '.join(eda_df[eda_df['label'] == 0]['text']).split()
ham_counts = Counter(ham_words).most_common(10)

# Extract and count words for SPAM (label == 1)
spam_words = ' '.join(eda_df[eda_df['label'] == 1]['text']).split()
spam_counts = Counter(spam_words).most_common(10)

print("Top 10 words in HAM messages:")
for word, count in ham_counts:
    print(f"{word}: {count}")
    
print("\nTop 10 words in SPAM messages:")
for word, count in spam_counts:
    print(f"{word}: {count}")

Top 10 words in HAM messages:
â: 200
state: 117
pm: 97
would: 94
ã: 89
president: 89
mr: 89
time: 81
percent: 80
obama: 77

Top 10 words in SPAM messages:
money: 847
account: 743
bank: 646
u: 631
fund: 626
e: 509
transaction: 471
business: 424
mr: 422
country: 422


## Extra features

In [12]:
data_train = pd.DataFrame({'preprocessed_text': X_train, 'label': y_train})
data_val = pd.DataFrame({'preprocessed_text': X_test, 'label': y_test})

# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_symbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_symbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_symbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,preprocessed_text,label,money_mark,suspicious_words,text_len
29,regard mr nelson smith kindly reply private em...,1,0,0,79
535,able reach oscar supposed send pdb receive,0,0,0,42
695,huma abedin checking pat work jack jake rest a...,0,0,0,76
557,announced monday today,0,0,0,22
836,bank africaagence san pedro bp san pedro cote ...,1,1,1,1104


## How would the Bag of Words work with Count Vectorizer concept?

In [13]:
from sklearn.feature_extraction.text import CountVectorizer
count_vectorizer = CountVectorizer()

X_train_bow = count_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_bow = count_vectorizer.transform(data_val['preprocessed_text'])

print("Bag of Words - Train shape:", X_train_bow.shape)
print("Bag of Words - Validation shape:", X_val_bow.shape)

Bag of Words - Train shape: (800, 28173)
Bag of Words - Validation shape: (200, 28173)


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vectorized dataset

In [14]:
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_tfidf = tfidf_vectorizer.transform(data_val['preprocessed_text'])

print("TF-IDF - Train shape:", X_train_tfidf.shape)
print("TF-IDF - Validation shape:", X_val_tfidf.shape)

TF-IDF - Train shape: (800, 28173)
TF-IDF - Validation shape: (200, 28173)


## And the Train a Classifier?

In [15]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

classifier = MultinomialNB()

classifier.fit(X_train_tfidf, data_train['label'])

y_pred = classifier.predict(X_val_tfidf)

print("Model Accuracy:", accuracy_score(data_val['label'], y_pred))
print("\nClassification Report:\n", classification_report(data_val['label'], y_pred))

Model Accuracy: 0.935

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.90      0.95       125
           1       0.85      1.00      0.92        75

    accuracy                           0.94       200
   macro avg       0.93      0.95      0.93       200
weighted avg       0.94      0.94      0.94       200



### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
import scipy.sparse as sp
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

# Prepare the extra flags/features
# We use MinMaxScaler so that the text_len values remain poisitive for MultinomialNB, which requires non-negative features
scaler = MinMaxScaler()
train_len_scaled = scaler.fit_transform(data_train[['text_len']])
val_len_scaled = scaler.transform(data_val[['text_len']])

# Combine binary flags and scaled length into dense arrays
extra_train = np.hstack([data_train[['money_mark', 'suspicious_words']].values, train_len_scaled])
extra_val = np.hstack([data_val[['money_mark', 'suspicious_words']].values, val_len_scaled])

clf = MultinomialNB()



# Scenario 1: Bag of Words Only
clf.fit(X_train_bow, data_train['label'])
acc_bow = accuracy_score(data_val['label'], clf.predict(X_val_bow))



# Scenario 2: TF-IDF Only
clf.fit(X_train_tfidf, data_train['label'])
acc_tfidf = accuracy_score(data_val['label'], clf.predict(X_val_tfidf))



# Scenario 3: Bag of Words + Extra Flags/Features
# Use sparse hstack to glue the text matrix and extra features together
X_train_bow_ext = sp.hstack([X_train_bow, extra_train])
X_val_bow_ext = sp.hstack([X_val_bow, extra_val])

clf.fit(X_train_bow_ext, data_train['label'])
acc_bow_ext = accuracy_score(data_val['label'], clf.predict(X_val_bow_ext))



# Scenario 4: TF-IDF + Extra Flags/Features
X_train_tfidf_ext = sp.hstack([X_train_tfidf, extra_train])
X_val_tfidf_ext = sp.hstack([X_val_tfidf, extra_val])

clf.fit(X_train_tfidf_ext, data_train['label'])
acc_tfidf_ext = accuracy_score(data_val['label'], clf.predict(X_val_tfidf_ext))




print("Model Accuracy comparison:")
print(f"1. Bag of Words Only: {acc_bow:.4f}")
print(f"2. TF_IDF Only: {acc_tfidf:.4f}")
print(f"3. Bag of Words + Extra Flags/Features: {acc_bow_ext:.4f}")
print(f"4. TF_IDF + Extra Flags/Features: {acc_tfidf_ext:.4f}")

Model Accuracy comparison:
1. Bag of Words Only: 0.9450
2. TF_IDF Only: 0.9350
3. Bag of Words + Extra Flags/Features: 0.9500
4. TF_IDF + Extra Flags/Features: 0.9200
